In [4]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip install "numpy<2" numba==0.63.1

Found existing installation: torch 2.8.0
Uninstalling torch-2.8.0:
  Successfully uninstalled torch-2.8.0
Found existing installation: torchvision 0.18.1
Uninstalling torchvision-0.18.1:
  Successfully uninstalled torchvision-0.18.1
Found existing installation: torchaudio 2.8.0
Uninstalling torchaudio-2.8.0:
  Successfully uninstalled torchaudio-2.8.0
  Using cached torch-2.3.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.18.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.3.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.4 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nv

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [1]:
import torch
import torchvision
import torchaudio

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)

torch: 2.3.1+cu121
torchvision: 0.18.1+cu121
torchaudio: 2.3.1+cu121


In [2]:
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install py-espeak-ng
!pip install phonemizer

In [3]:
!pip show whisperx

Name: whisperx
Version: 3.8.1
Summary: Time-Accurate Automatic Speech Recognition using Whisper.
Home-page: 
Author: Max Bain
Author-email: 
License: BSD-2-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: ctranslate2, faster-whisper, huggingface-hub, nltk, numpy, omegaconf, pandas, pyannote-audio, torch, torchaudio, transformers, triton
Required-by: 


In [4]:
from google.colab import drive
drive.mount('/content/drive')

audio_file_path = '/content/drive/MyDrive/Colab Notebooks/hedge.mp3'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from phonemizer import phonemize
text_to_phonemize = "hello world! this is a test."

phonemes = phonemize(text_to_phonemize, language='en-us', backend='espeak', strip=True)
print(phonemes)

həloʊ wɜːld ðɪs ɪz ɐ tɛst


In [15]:
import whisperx

device = "cpu"
audio_file = audio_file_path
batch_size = 16 # reduce if low on GPU mem
compute_type = "int8" # change to "int8" if low on GPU mem (may reduce accuracy)

# 1. Transcribe with original whisper (batched)
# model = whisperx.load_model("large-v2", device, compute_type=compute_type)
model = whisperx.load_model("small", device, compute_type=compute_type, language="en")

audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
transcript = model.transcribe(audio, batch_size=batch_size)["segments"]
print(transcript) # before alignment

2026-02-16 04:33:41 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


[{'text': " Fine, you worry about your casserole and I'll worry about the end of suburban peace and tranquility!", 'start': 0.031, 'end': 5.971}]


In [16]:
# Use phonemize to get the transcript in terms of phonemes
phone_transcript = [{"text": phonemize(segment["text"]), "start":segment["start"], "end":segment["end"]} for segment in transcript]

print(phone_transcript) # 'aɪ noʊ naʊ wʌt bɹɪŋz miː tə ðə paɪɚ'

# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
print(metadata)

result = whisperx.align(phone_transcript, model_a, metadata, audio, device, return_char_alignments=True)

print(result["segments"]) # after alignment

[{'text': 'faɪn juː wʌɹi ɐbaʊt jʊɹ kæsɚɹoʊl ænd aɪl wʌɹi ɐbaʊt ðɪ ɛnd ʌv səbɜːbən piːs ænd tɹæŋkwɪlᵻɾi ', 'start': 0.031, 'end': 5.971}]
{'language': 'en', 'dictionary': {'-': 0, '|': 1, 'e': 2, 't': 3, 'a': 4, 'o': 5, 'n': 6, 'i': 7, 'h': 8, 's': 9, 'r': 10, 'd': 11, 'l': 12, 'u': 13, 'm': 14, 'w': 15, 'c': 16, 'f': 17, 'g': 18, 'y': 19, 'p': 20, 'b': 21, 'v': 22, 'k': 23, "'": 24, 'x': 25, 'j': 26, 'q': 27, 'z': 28}, 'type': 'torchaudio'}
[{'start': 0.031, 'end': 5.991, 'text': 'faɪn juː wʌɹi ɐbaʊt jʊɹ kæsɚɹoʊl ænd aɪl wʌɹi ɐbaʊt ðɪ ɛnd ʌv səbɜːbən piːs ænd tɹæŋkwɪlᵻɾi', 'words': [{'word': 'faɪn', 'start': 0.031, 'end': 0.637, 'score': np.float64(0.83)}, {'word': 'juː', 'start': 0.839, 'end': 0.92, 'score': np.float64(0.993)}, {'word': 'wʌɹi', 'start': 0.94, 'end': 1.122, 'score': np.float64(0.847)}, {'word': 'ɐbaʊt', 'start': 1.142, 'end': 1.324, 'score': np.float64(0.821)}, {'word': 'jʊɹ', 'start': 1.385, 'end': 1.506, 'score': np.float64(0.753)}, {'word': 'kæsɚɹoʊl', 'start': 1.56